# Hands-on Task: Reliable Tool Caller

Define one function schema and get a model to call it correctly with valid JSON arguments, 5 times in a row. This notebook walks you through building `ToolCallTester`, step by step, using everything from Concepts 1–4.

## Setup

In [ ]:
!pip install -q openai jsonschema

In [ ]:
import os

import json

from openai import OpenAI

from jsonschema import validate, ValidationError



# os.environ["OPENAI_API_KEY"] = "sk-..."  # uncomment and set if not already in your environment

client = OpenAI()

## Step 1: Define the Schema
One function, `get_weather(city, unit)` — the same one used throughout this module.

In [ ]:
TOOL_NAME = "get_weather"



get_weather_schema = {

    "type": "object",

    "properties": {

        "city": {"type": "string"},

        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}

    },

    "required": ["city", "unit"]

}



tools = [{

    "type": "function",

    "function": {

        "name": TOOL_NAME,

        "description": "Get current weather for a city",

        "parameters": get_weather_schema

    }

}]

## Step 2 & 3: Call the Model and Validate
`call_once` sends a prompt, extracts the tool call, validates the arguments, and retries with the error fed back if validation fails.

In [ ]:
class ToolCallTester:

    def __init__(self, client, tools, tool_name, schema):

        self.client = client

        self.tools = tools

        self.tool_name = tool_name

        self.schema = schema



    def call_once(self, prompt, max_retries=3):

        messages = [{"role": "user", "content": prompt}]

        for attempt in range(1, max_retries + 1):

            response = self.client.chat.completions.create(

                model="gpt-4o-mini",

                messages=messages,

                tools=self.tools,

                temperature=0,

            )

            message = response.choices[0].message

            tool_calls = message.tool_calls or []

            if not tool_calls:

                messages.append({"role": "user", "content": f"Please call the {self.tool_name} tool."})

                continue



            call = tool_calls[0]

            try:

                args = json.loads(call.function.arguments)

                validate(instance=args, schema=self.schema)

                return args, attempt

            except (json.JSONDecodeError, ValidationError) as e:

                messages.append({"role": "assistant", "content": None, "tool_calls": [call]})

                messages.append({

                    "role": "tool",

                    "tool_call_id": call.id,

                    "content": f"Invalid arguments: {e}. Please call {self.tool_name} again with valid arguments."

                })

        raise RuntimeError(f"Failed to get valid arguments for prompt: {prompt!r}")



    def run_suite(self, prompts, max_retries=3):

        results = []

        for prompt in prompts:

            try:

                args, attempts = self.call_once(prompt, max_retries=max_retries)

                results.append({"prompt": prompt, "args": args, "attempts": attempts, "success": True})

            except RuntimeError as e:

                results.append({"prompt": prompt, "error": str(e), "success": False})

        return results

## Step 4: Run the Suite
Five indirect prompts — none of them literally ask for the weather, but each one needs it to answer well. This is the realistic case: the model has to *decide* a tool is needed, not just fill in a form for an obvious question.

In [ ]:
prompts = [
    "I'm travelling to Pune tomorrow, what should I pack? Use celsius.",
    "Should I carry an umbrella if I'm heading out in Tokyo today? Fahrenheit please.",
    "Deciding between a t-shirt and a sweater for my walk in Delhi later, in celsius.",
    "I'm hosting an outdoor lunch in London tomorrow, should I move it indoors? I'm using fahrenheit.",
    "Heading out in Mumbai this evening, do I need a jacket? Celsius please.",
]

tester = ToolCallTester(client, tools, TOOL_NAME, get_weather_schema)
results = tester.run_suite(prompts)

for r in results:
    if r["success"]:
        print(f"OK  (attempt {r['attempts']}): {r['prompt']!r} -> {r['args']}")
    else:
        print(f"FAIL: {r['prompt']!r} -> {r['error']}")

## Step 5: Report Your Results

In [ ]:
success_count = sum(1 for r in results if r["success"])

total_attempts = sum(r["attempts"] for r in results if r["success"])

print(f"Success rate: {success_count}/{len(prompts)}")

print(f"Total attempts used: {total_attempts} (retries: {total_attempts - success_count})")

## Bonus: Add Basic Smoothing... err, Reliability
Try lowering `max_retries` to 1 and see how many prompts fail on the first try. Then try raising the tool's temperature away from 0 and see if reliability drops.

## Success Criteria
You've completed this task when `success_count == 5` — five prompts, five valid, schema-conformant results, regardless of how many retries each one needed.